We will be mainly using pandas and numpy for the analysis of the data provided

In [1]:
import pandas as pd
import numpy as np

power = pd.read_csv('PGCB_date_power_demand.csv')
power = power[['datetime','demand_mw']]

power.head()

,datetime,demand_mw
0,2015-04-19 22:00:00,6323
1,2015-04-19 21:00:00,6667
2,2015-04-19 19:00:00,6897
3,2015-04-19 18:30:00,6933
4,2015-04-19 18:00:00,6874


In [2]:
power['datetime'] = pd.to_datetime(power['datetime']) #convert the string datetime to pandas datetime
power.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92650 entries, 0 to 92649
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   datetime   92650 non-null  datetime64[ns]
 1   demand_mw  92650 non-null  int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 1.4 MB


In [3]:
weather = pd.read_csv('weather_data.csv')
#renaming the column names to access them later
weather.rename(columns={'latitude':'time',
                        'utc_offset_seconds':'apparent_temperature (°C)',
                        'timezone':'precipitation (mm)',
                        'Unnamed: 9': 'sunshine_duration (s)',
                        'elevation' : 'relative_humidity_2m (%)' }, inplace=True)

#keeping only the logically useful columns
weather = weather[['time', 'apparent_temperature (°C)', 'precipitation (mm)','sunshine_duration (s)','relative_humidity_2m (%)']]
weather = weather.iloc[3:].reset_index(drop=True) #removing the first 3 rows as they are useless
weather['time'] = pd.to_datetime(weather['time']) #convert the string datetime to pandas datetime

weather.head()

C:\Users\yashc\AppData\Local\Temp\ipykernel_1728\254298203.py:1: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  weather = pd.read_csv('weather_data.csv')


,time,apparent_temperature (°C),precipitation (mm),sunshine_duration (s),relative_humidity_2m (%)
0,2014-01-01 00:00:00,13.3,0,0,89
1,2014-01-01 01:00:00,13.2,0,0,91
2,2014-01-01 02:00:00,12.8,0,0,91
3,2014-01-01 03:00:00,12.5,0,0,92
4,2014-01-01 04:00:00,12.2,0,0,93


We kept only temperature, precipitation, relative humidity and sunshine duration as they may affect the power consumption

In [4]:
economic = pd.read_csv('economic_full_1.csv')
required_indicators = [
    'Urban population',
    'Rural population',
    'Electric power consumption (kWh per capita)'
]
economic = economic[economic['Indicator Name'].isin(required_indicators)]
economic.head()

,Country Name,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
25,X,Urban population,SP.URB.TOTL,2649573.0,2813246.0,2999574.0,3195917.0,3406306.0,3633117.0,3878538.0,...,4.804319e+07,4.898278e+07,4.990327e+07,5.083089e+07,5.174678e+07,5.264356e+07,5.365052e+07,55140091.0,56717264.0,NaN
26,X,Rural population,SP.RUR.TOTL,49179087.0,50497102.0,51881572.0,53308485.0,54772068.0,56269402.0,57791268.0,...,1.127687e+08,1.132034e+08,1.136198e+08,1.140822e+08,1.145512e+08,1.150153e+08,1.157344e+08,116326899.0,116845100.0,NaN
1494,X,Electric power consumption (kWh per capita),EG.USE.ELEC.KH.PC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.127803e+02,4.432129e+02,4.711261e+02,5.151017e+02,5.099459e+02,5.741182e+02,6.026747e+02,NaN,NaN,NaN


from the economic indicator dataset we only took 3 indicators, because i found only these 3 useful regarding our prediction and i am using the last year data as the feature to prevent data leak in the model

In [5]:
#extract only the required year columns
year_cols = [str(year) for year in range(2013, 2023)]

economic = economic[['Indicator Name'] + year_cols]
economic.rename(columns={'2013':'2014',
                         '2014':'2015',
                         '2015':'2016',
                         '2016':'2017',
                         '2017':'2018',
                         '2018':'2019',
                         '2019':'2020',
                         '2020':'2021',
                         '2021':'2022',
                         '2022':'2023'}, inplace=True)
economic.reset_index(inplace=True)
economic.drop(columns=['index'], inplace=True)
economic.head()

,Indicator Name,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Urban population,4.514497e+07,4.611698e+07,4.708101e+07,4.804319e+07,4.898278e+07,4.990327e+07,5.083089e+07,5.174678e+07,5.264356e+07,5.365052e+07
1,Rural population,1.113872e+08,1.118479e+08,1.123022e+08,1.127687e+08,1.132034e+08,1.136198e+08,1.140822e+08,1.145512e+08,1.150153e+08,1.157344e+08
2,Electric power consumption (kWh per capita),2.963417e+02,3.265029e+02,3.489327e+02,4.127803e+02,4.432129e+02,4.711261e+02,5.151017e+02,5.099459e+02,5.741182e+02,6.026747e+02


changing the index of the economic data to the datetime as the PGCB dataset and weather dataset to merge all three and also we use forward fiil to fill the missing values in the economic dataset.

In [6]:
expanded_data = []
year_cols = [str(year) for year in range(2015, 2024)]
for _, row in economic.iterrows():
    indicator = row['Indicator Name']

    for year in year_cols:
        value = row[year]

        date_range = pd.date_range(
            start=f"{year}-01-01 00:00:00",
            end=f"{year}-12-31 23:00:00",
            freq='h'
        )

        temp_df = pd.DataFrame({
            'datetime': date_range,
            'Indicator Name': indicator,
            'value': value
        })

        expanded_data.append(temp_df)

economic_expanded = pd.concat(expanded_data, ignore_index=True)

economic_expanded = economic_expanded.sort_values(['Indicator Name', 'datetime'])
economic_expanded['value'] = economic_expanded.groupby('Indicator Name')['value'].ffill()
print(economic_expanded.isna().sum())
economic_expanded.head()
economic_expanded.head()

datetime          0
Indicator Name    0
value             0
dtype: int64


,datetime,Indicator Name,value
157776,2015-01-01 00:00:00,Electric power consumption (kWh per capita),326.502853
157777,2015-01-01 01:00:00,Electric power consumption (kWh per capita),326.502853
157778,2015-01-01 02:00:00,Electric power consumption (kWh per capita),326.502853
157779,2015-01-01 03:00:00,Electric power consumption (kWh per capita),326.502853
157780,2015-01-01 04:00:00,Electric power consumption (kWh per capita),326.502853


Merging the economic and the weather dataset into a single dataset, and then calculating the correlation between features and the target variable

In [7]:
economic_expanded['datetime'] = pd.to_datetime(economic_expanded['datetime'])
merged = pd.merge(
    economic_expanded,
    weather,
    left_on='datetime',
    right_on='time',
    how='inner'
)

merged_pivot = merged.pivot_table(
    index='datetime',
    columns='Indicator Name',
    values='value'
).reset_index()

final_df = pd.merge(
    merged_pivot,
    weather,
    left_on='datetime',
    right_on='time',
    how='inner'
)

final_df = final_df.drop(columns=['time'])# dropping the duplicate columns

final_df = pd.merge(
    final_df,
    power[['datetime', 'demand_mw']],
    on='datetime',
    how='inner'
)

corr_matrix = final_df.drop(columns=['datetime']).corr()
demand_corr = corr_matrix['demand_mw'].sort_values(ascending=False)

print(demand_corr)
print(final_df)

demand_mw                                      1.000000
Rural population                               0.641009
Urban population                               0.640155
Electric power consumption (kWh per capita)    0.630146
apparent_temperature (°C)                      0.471343
sunshine_duration (s)                         -0.012052
precipitation (mm)                            -0.027308
relative_humidity_2m (%)                      -0.044948
Name: demand_mw, dtype: float64
                 datetime  Electric power consumption (kWh per capita)  \
0     2015-04-19 00:00:00                                   326.502853   
1     2015-04-19 01:00:00                                   326.502853   
2     2015-04-19 02:00:00                                   326.502853   
3     2015-04-19 03:00:00                                   326.502853   
4     2015-04-19 04:00:00                                   326.502853   
...                   ...                                          ...   
75

we observe that the corr of rural population, urban population,Electric power consumption, apparent temperature is really high means they all are important features

Now we will create the lag features to help the model see back in time and help predict the next hour power consumption


In [8]:
final_df = final_df.sort_values('datetime')
final_df = final_df.set_index('datetime')

#target column
col = 'demand_mw'

#lag features
final_df['lag_1'] = final_df[col].shift(1)
final_df['lag_6'] = final_df[col].shift(6)
final_df['lag_12'] = final_df[col].shift(12)
final_df['lag_24'] = final_df[col].shift(24)
final_df['lag_72'] = final_df[col].shift(72)
final_df['lag_144'] = final_df[col].shift(144)

final_df['lag_1r'] = final_df[col].shift(1).rolling(1).mean()
final_df['lag_6r'] = final_df[col].shift(1).rolling(6).mean()
final_df['lag_12r'] = final_df[col].shift(1).rolling(12).mean()
final_df['lag_24r'] = final_df[col].shift(1).rolling(24).mean()
final_df['lag_72r'] = final_df[col].shift(1).rolling(72).mean()
final_df['lag_144r'] = final_df[col].shift(1).rolling(144).mean()

final_df = final_df.reset_index()

here lag_1 denotes the demand of last hour, lag_6 denotes the demand 6 hours ago and soo on

we now again find the corr of the lag features with the target variable

In [9]:
corr_matrix = final_df.drop(columns=['datetime']).corr()
demand_corr = corr_matrix['demand_mw'].sort_values(ascending=False)

print(demand_corr)

demand_mw                                      1.000000
lag_1                                          0.922565
lag_1r                                         0.922565
lag_6r                                         0.890525
lag_24                                         0.884885
lag_24r                                        0.874879
lag_72r                                        0.857450
lag_12r                                        0.852247
lag_144r                                       0.843089
lag_72                                         0.839409
lag_144                                        0.809429
lag_6                                          0.771522
lag_12                                         0.699180
Rural population                               0.641009
Urban population                               0.640155
Electric power consumption (kWh per capita)    0.630146
apparent_temperature (°C)                      0.471343
sunshine_duration (s)                         -0

Notice that the corr of all the lag features is really high this means that our approach of using them to find the demand is correct

we drop relative humidity, sunshine duration and precipitation as they are not really well related with the target variable(low corr)

In [10]:
final_df.drop(columns=['relative_humidity_2m (%)', 'sunshine_duration (s)', 'precipitation (mm)', 'datetime'], inplace=True)

convert the apparent temperature to numeric datatype

In [11]:
final_df['apparent_temperature (°C)'] = pd.to_numeric(
    final_df['apparent_temperature (°C)'],
    errors='coerce'
)

Now we go forward with model training
first we prepare the input and the target dataset

In [12]:
TARGET = 'demand_mw'

X = final_df.drop(columns=[TARGET])#input
y = final_df[TARGET]#target

we will split the dataset into training and the evaluation datasets, we use 0.9 as we will use years from 2014-2022 (9 years) to train the model and year 2023(one year) to evaluate the model

In [13]:
split = int(len(final_df) * 0.9)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

As for the model we will be using xgboost as it handles non-linear relationships extremely well, is robust to outliers and has high predictive accuracy

In [14]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [15]:
y_pred = model.predict(X_test)

calculating the MAPE score

In [20]:
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
print("MAPE:", mape)

MAPE: 5.077211558704382


we get a MAPE of 5.07 which is good for our use case

finally we see which feature have played major role in the prediction of the target variable

In [17]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance.head(15))

lag_1r                                         0.577955
lag_1                                          0.321600
lag_24                                         0.016591
Urban population                               0.015916
Rural population                               0.012974
Electric power consumption (kWh per capita)    0.007961
apparent_temperature (°C)                      0.007325
lag_144                                        0.005896
lag_72                                         0.005519
lag_6                                          0.005500
lag_12                                         0.004846
lag_12r                                        0.004719
lag_6r                                         0.004517
lag_24r                                        0.003257
lag_144r                                       0.002858
dtype: float32


we see that lag_1r act as the major feature in deciding the demand in the next hour, while rural and urban population had good corr they don't contribute that much in next hour power supply which is understood logically also.